In [ ]:
import pandas as pd
from pathlib import Path

# 1. قراءة الملف
manifest_cleaned = pd.read_csv(r"C:\BCI\cortex-call\data\processed\manifest_clean.csv")

# 2. إزالة كلمة 'notebooks\' من المسارات إذا كانت موجودة
manifest_cleaned['file_path'] = manifest_cleaned['file_path'].str.replace(r'notebooks\\', '', regex=False)
manifest_cleaned['file_path'] = manifest_cleaned['file_path'].str.replace('notebooks/', '', regex=False)

# 3. حفظ الملف بعد التعديل الصحيح
manifest_cleaned.to_csv(r"C:\BCI\cortex-call\data\processed\manifest_clean.csv", index=False)

# 4. طباعة النتيجة للتأكد
print(manifest_cleaned.head())

In [22]:
import pandas as pd
from pathlib import Path

manifest_cleaned = pd.read_csv(r"C:\BCI\cortex-call\data\processed\manifest_clean.csv")
print(manifest_cleaned.head())

   trial_id                                          file_path  label  \
0         1  C:\BCI\cortex-call\data\processed\filtered_tri...   left   
1         2  C:\BCI\cortex-call\data\processed\filtered_tri...  right   
2         3  C:\BCI\cortex-call\data\processed\filtered_tri...   left   
3         4  C:\BCI\cortex-call\data\processed\filtered_tri...   left   
4         5  C:\BCI\cortex-call\data\processed\filtered_tri...   left   

   rejected  
0     False  
1     False  
2     False  
3     False  
4     False  


In [18]:
# --- 1) Verify if the manifest DataFrame is sorted ascending by trial_id ---
is_manifest_sorted = manifest_cleaned["trial_id"].is_monotonic_increasing

# Output the boolean result of the monotonic check
print(f"Manifest sorted ascending by trial_id: {is_manifest_sorted}")

Manifest sorted ascending by trial_id: True


In [25]:
# Initialize an empty list to store trial DataFrames
frames = []

# Iterate over each trial entry in the validated manifest
for _, row in manifest_cleaned.iterrows():
    # Load raw EEG signals for the current trial file
    if row['rejected'] == False:
        
        df = pd.read_csv(row["file_path"])

    # Annotate DataFrame with trial metadata
        df["trial_id"] = row["trial_id"]
        df["label"] = row["label"]

    # Append individual trial frame to collection
    frames.append(df)

# Concatenate all trial frames into a unified master EEG DataFrame
full_eeg_df = pd.concat(frames, ignore_index=True)

# Output summary dimensions and preview initial rows
print(f"full_eeg_df shape: {full_eeg_df.shape}")
full_eeg_df.head()

full_eeg_df shape: (4987500, 7)


,Time,FZ,C3,CZ,C4,trial_id,label
0,0.000,57.027373,36.694743,-19.247874,-213.433761,1,left
1,0.004,79.773387,12.120732,-238.491398,-781.799151,1,left
2,0.008,72.283575,-21.739405,-380.349589,-1081.032070,1,left
3,0.012,45.618464,-47.003288,-399.507800,-1044.052597,1,left
4,0.016,36.514681,-38.474558,-319.470402,-839.499283,1,left


In [27]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from mne.decoding import CSP

# 1. تحديد القنوات
channels = ['FZ', 'C3', 'CZ', 'C4']

# 2. إيجاد الحد الأدنى من النقاط الزمنية لتثبيت طول الـ Trials
min_timepoints = full_eeg_df.groupby('trial_id').size().min()

# 3. إعادة تشكيل البيانات إلى 3D Array: (n_trials, n_channels, n_times)
X_list = []
y_list = []

for trial_id, group in full_eeg_df.groupby('trial_id'):
    # إشارة القنوات مفصولة للطول المحدد (n_times, n_channels)
    signal = group[channels].values[:min_timepoints, :]
    
    # نقل المحاور لتببح (n_channels, n_times)
    signal = signal.T 
    
    X_list.append(signal)
    y_list.append(group['label'].iloc[0])

X = np.array(X_list)  # الشكل النهائي: (n_trials, 4, min_timepoints)
y = np.array(y_list)

print(f"شكل مصفوفة البيانات (Trials, Channels, Timepoints): {X.shape}")

# 4. تقسيم البيانات على مستوى الـ Trials
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. بناء الأنبوب (Pipeline): CSP -> Scaler -> SVM
# n_components: عدد الأنماط المكانية المستخرجة (يجب أن يكون أقل أو يساوي عدد القنوات)
csp = CSP(n_components=4, log=True, norm_trace=False)

svm_model = SVC(kernel='rbf', C=1.0, random_state=42)

# تجميع الخطوات في Pipeline لضمان عدم حدوث Data Leakage
pipeline = Pipeline([
    ('CSP', csp),
    ('Scaler', StandardScaler()),
    ('SVM', svm_model)
])

# 6. التدريب والتقييم
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f"\nAccuracy Score with CSP + SVM: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

شكل مصفوفة البيانات (Trials, Channels, Timepoints): (1995, 4, 2500)
Computing rank from data with rank=None
    Using tolerance 1.1e+03 (2.2e-16 eps * 4 dim * 1.2e+18  max singular value)
    Estimated rank (data): 4
    data: rank 4 computed from 4 data channels with 0 projectors
Reducing data rank from 4 -> 4
Estimating class=left covariance using EMPIRICAL
Done.
Estimating class=right covariance using EMPIRICAL
Done.

Accuracy Score with CSP + SVM: 52.38%

Classification Report:
              precision    recall  f1-score   support

        left       0.52      0.66      0.58       199
       right       0.53      0.39      0.45       200

    accuracy                           0.52       399
   macro avg       0.53      0.52      0.52       399
weighted avg       0.53      0.52      0.52       399

